# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamikshaBurte/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
# Setup & Repo Cloning for Colab
import os, sys, subprocess

REPO_URL = "https://github.com/SamikshaBurte/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")

print("Current Working Directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check folder path"
print("Repo setup complete! CSV found.")

Current Working Directory: /content/flyrank-ml-internship
Repo setup complete! CSV found.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify class balance of the target classification variable
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

class_counts = df['is_declining'].value_counts()
class_percentages = df['is_declining'].value_counts(normalize=True) * 100

print("=== Target Class Distribution ===")
print(f"Non-Declining (0) : {class_counts[0]:,} ({class_percentages[0]:.2f}%)")
print(f"Declining (1)     : {class_counts[1]:,} ({class_percentages[1]:.2f}%)")

=== Target Class Distribution ===
Non-Declining (0) : 13,738 (45.79%)
Declining (1)     : 16,262 (54.21%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect target column creation and sample feature matrix
df['target'] = (df['trend_direction'] == 'down').astype(int)

# Pick whichever identifier column exists in the CSV
id_col = 'url_hash' if 'url_hash' in df.columns else ('url' if 'url' in df.columns else df.columns[0])

feature_cols = [id_col, 'impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'target']
sample_df = df[[col for col in feature_cols if col in df.columns]].head(5)

print("=== Sample Target Mapping ===")
print(sample_df.to_string(index=False))

=== Sample Target Mapping ===
          content_id  impressions_90d  ctr  avg_position  content_age_days  target
content_304f48230142             3803 0.76          10.6               187       1
content_a1fb4e703a9e            15320 0.05          20.3               445       1
content_9aa793d4d895            12581 0.09          36.5               141       1
content_331d6c4de07b            11751 0.49           6.2               463       0
content_d99b7a2d90ca            19140 0.13          44.0               263       1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

# Baseline Precision@50 evaluation function
def calculate_precision_at_k(y_true, scores, k=50):
    top_k_indices = np.argsort(-scores)[:k]
    return y_true.iloc[top_k_indices].mean()

# Heuristic Score: High age + low CTR
df['heuristic_score'] = df['content_age_days'] * (1 / (df['ctr'] + 0.001))
baseline_p50 = calculate_precision_at_k(df['target'], df['heuristic_score'], k=50)

print(f"Baseline Heuristic Precision@50: {baseline_p50 * 100:.2f}%")

Baseline Heuristic Precision@50: 80.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show unit of analysis shape and schema
print(f"Unit of Analysis DataFrame Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\n=== Key Feature Data Types & Non-Null Counts ===")
feature_cols = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'target']
print(df[feature_cols].info())

Unit of Analysis DataFrame Shape: 30,000 rows x 47 columns

=== Key Feature Data Types & Non-Null Counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   impressions_90d   30000 non-null  int64  
 1   clicks_90d        30000 non-null  int64  
 2   ctr               30000 non-null  float64
 3   avg_position      30000 non-null  float64
 4   content_age_days  30000 non-null  int64  
 5   target            30000 non-null  int64  
dtypes: float64(2), int64(4)
memory usage: 1.4 MB
None


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrating non-linear complexity: CTR varies widely across position tiers
df['position_tier'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 100], labels=['Top 3', 'Positions 4-10', 'Page 2+'])
ctr_summary = df.groupby(['position_tier', 'target'], observed=False)['ctr'].mean().unstack()
ctr_summary.columns = ['Stable/Growing CTR', 'Declining CTR']

print("=== Average CTR by Position Tier & Target Status ===")
print(ctr_summary.round(4))

=== Average CTR by Position Tier & Target Status ===
                Stable/Growing CTR  Declining CTR
position_tier                                    
Top 3                       4.2586         1.1565
Positions 4-10              0.9376         0.4344
Page 2+                     0.3605         0.1885


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.